In [1]:
import math
import random
import numpy as np
import time
from waveNetArchitecture import Value, Linear, BatchNorm1D, LayerNorm, Tanh, ReLU, Embedding, FlattenConsecutive, Sequential, cross_entropy, BagOfWords, PositionalEmbedding, Head, MultiHead, FeedForward, Block, AdamW
from bpeTokenizer import RegexTokenizer
with open('fineweb_edu_subset.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
tok = RegexTokenizer.load('shakespearTokenizer.json')
EOS_ID = tok.add_special_token('<|endoftext|>')  # already in the saved file, so this just looks the id up
# no tok.save() here - this notebook is a throwaway experiment and shouldn't rewrite shared files
vocab_size = tok.vocab_size
encode = tok.encode
decode = tok.decode

blockSize = 64 # same context as the real model, so the test runs through the same code paths
print(vocab_size, 'EOS_ID:', EOS_ID)

1025 EOS_ID: 1024


In [ ]:
overfit_docs = 1   # documents to keep. Doc 0 is ~1,600 tokens; 2 docs is ~3,900

docs = [d for d in text.split('\n\n') if d][:overfit_docs]
ids = []
for d in docs:
    ids.extend(encode(d))
    ids.append(EOS_ID)
data = np.array(ids) # plain int array: this is data, not a parameter, so it needs no autograd

# no val split - memorizing the training set IS the test, so all of it is train
train_data = data

# A model that ignores context entirely can only learn how common each token is, so
# its best possible loss is the unigram entropy of the data. Getting clearly below
# this line is proof that attention and positions are actually doing something.
counts = np.bincount(data, minlength=vocab_size)
p = counts[counts > 0] / len(data)
unigram_entropy = float(-(p * np.log(p)).sum())

print(f'docs: {len(docs)}   tokens: {len(data):,}   distinct tokens: {int((counts > 0).sum())}')

docs: 1   tokens: 1,621   distinct tokens: 384


In [ ]:

n_embd = 32     # embedding dimensionality (must divide evenly across num_heads)
num_heads = 4   # attention heads per block -> head_size 8
n_blocks = 4    # how many transformer blocks to stack

model = Sequential([
  Embedding(vocab_size, n_embd),
  PositionalEmbedding(blockSize, n_embd),
  *[Block(n_embd, num_heads, blockSize) for _ in range(n_blocks)],
  LayerNorm(n_embd),
  Linear(n_embd, vocab_size),
])

# keep initial predictions close to uniform, so the starting loss is near -log(1/vocab_size)
model.layers[-1].weight.data *= 0.1

parameters = model.parameters()
print(sum(p.data.size for p in parameters)) # number of parameters in total

119169


In [ ]:
from collections import defaultdict, Counter

max_steps = 2000
batch_size = 16
eval_every = 200

lossi = []
full_lossi = []
ud = []

opt = AdamW(parameters, lr=1e-3, betas=(0.9,0.95), weight_decay=0.01)

def make_windows(tokens, blockSize):
    n = len(tokens) - blockSize
    X = np.lib.stride_tricks.sliding_window_view(tokens, blockSize)[:n]      # (n, blockSize) context
    Y = np.lib.stride_tricks.sliding_window_view(tokens[1:], blockSize)[:n]  # (n, blockSize) same, shifted by one
    return X, Y

Xtr, Ytr = make_windows(train_data, blockSize)

def memorization_floor(X, Y):
    groups = defaultdict(Counter)
    for x, y in zip(X, Y):
        for t in range(len(x)):
            groups[x[:t+1].tobytes()][int(y[t])] += 1
    nll, total = 0.0, 0
    for c in groups.values():
        n = sum(c.values())
        for k in c.values():
            nll -= k * np.log(k / n)
        total += n
    return nll / total

floor = memorization_floor(Xtr, Ytr)
print(f'train windows: {Xtr.shape[0]:,}')
print(f'  {np.log(vocab_size):.3f}  untrained (uniform over the vocab)')
print(f'  {unigram_entropy:.3f}  best a context-blind model can do  <- must get well below this')
print(f'  {floor:.3f}  perfect memorization')

def full_loss(X, Y, chunk=32):
    total = 0.0
    for s in range(0, X.shape[0], chunk):
        xb, yb = X[s:s+chunk], Y[s:s+chunk]
        total += float(cross_entropy(model(xb), yb).data) * xb.shape[0]
    return total / X.shape[0]

t0 = time.time()
for i in range(max_steps):

    ix = np.random.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]  # (batch_size, blockSize), (batch_size, blockSize)

    logits = model(Xb)               # (batch_size, blockSize, vocab_size)
    loss = cross_entropy(logits, Yb)

    opt.zero_grad()
    loss.backward()

    lr = 1e-3 if i < 0.8 * max_steps else 1e-4
    opt.step(lr)

    lossi.append(float(loss.data))
    ud.append(opt.ud)

    if i % 25 == 0:
        print(f'{i:6d}/{max_steps}: {np.mean(lossi[-50:]):.4f}   ({time.time()-t0:.0f}s)')
    if i % eval_every == 0 or i == max_steps - 1:
        fl = full_loss(Xtr, Ytr)
        full_lossi.append((i, fl))
        print(f'{i:6d}/{max_steps}: FULL-DATA LOSS {fl:.4f}   '
              f'(context-blind line {unigram_entropy:.3f}, memorization floor {floor:.3f})')

train windows: 1,557
  6.932  untrained (uniform over the vocab)
  5.395  best a context-blind model can do  <- must get well below this
  0.031  perfect memorization
     0/2000: 6.9420   (3s)
     0/2000: FULL-DATA LOSS 6.9161   (context-blind line 5.395, memorization floor 0.031)
    25/2000: 6.5407   (158s)
    50/2000: 6.1850   (243s)
    75/2000: 5.6881   (323s)
   100/2000: 5.4552   (405s)
   125/2000: 5.3089   (489s)
   150/2000: 5.1331   (557s)
   175/2000: 4.8978   (648s)
   200/2000: 4.6549   (737s)
   200/2000: FULL-DATA LOSS 4.4002   (context-blind line 5.395, memorization floor 0.031)
   225/2000: 4.4026   (996s)
   250/2000: 4.1539   (1072s)
   275/2000: 3.9182   (1138s)
   300/2000: 3.6801   (1213s)
   325/2000: 3.4681   (1292s)
   350/2000: 3.2753   (1331s)
   375/2000: 3.0970   (1402s)
   400/2000: 2.9222   (1475s)
   400/2000: FULL-DATA LOSS 2.7324   (context-blind line 5.395, memorization floor 0.031)
   425/2000: 2.7421   (1676s)
   450/2000: 2.5763   (1757s)
   47

In [ ]:
# THE ACTUAL TEST: can it reproduce the text it was trained on?

correct = total = 0
ctx_correct = ctx_total = 0   # only positions with 16+ tokens of context, where ambiguity is ~gone
for s in range(0, Xtr.shape[0], 32):
    xb, yb = Xtr[s:s+32], Ytr[s:s+32]
    pred = model(xb).data.argmax(axis=-1)
    correct += int((pred == yb).sum());                  total += yb.size
    ctx_correct += int((pred[:, 16:] == yb[:, 16:]).sum()); ctx_total += yb[:, 16:].size
print(f'teacher-forced next-token accuracy: {correct/total:.1%} overall, '
      f'{ctx_correct/ctx_total:.1%} with 16+ tokens of context')

prompt = [int(t) for t in train_data[:blockSize]]
truth = [int(t) for t in train_data[blockSize:]]
context = prompt[:]
generated = []
for _ in range(min(300, len(truth))):
    logits = model(np.array([context])).data[0, -1]
    ix = int(logits.argmax())
    generated.append(ix)
    context = context[1:] + [ix]

match = 0
for g, t in zip(generated, truth):
    if g != t:
        break
    match += 1
print(f'free-running recall: matched the real text for {match} of {len(generated)} tokens before the first mistake')
print('\n--- prompt (first blockSize tokens of the doc) ---\n' + decode(prompt))
print('\n--- model continues ---\n' + decode(generated))
print('\n--- actual text ---\n' + decode(truth[:len(generated)]))

teacher-forced next-token accuracy: 97.4% overall, 99.5% with 16+ tokens of context
free-running recall: matched the real text for 188 of 300 tokens before the first mistake

--- prompt (first blockSize tokens of the doc) ---
The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought

--- model continues ---
 and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and knowing her father would disapprove was unusual even for Austen. In her last book Annnne Elliot is persuaded to refuse Congress inte Ema.
Jollinenter rescatented to rescatedregagement to the rules of Massinent unlike any
Alth or there abo

In [7]:
# sample from the model
rng = np.random

out = []
context = [0] * blockSize
while len(out) < 300:  # each token is a full forward pass, so keep this short
    x = np.array([context]) # (1, blockSize); the Embedding layer looks these up
    logits = model(x)                     # (1, blockSize, vocab_size): a prediction at every position
    last_logits = logits.data[:, -1, :]    # only the newest position matters for generation
    e = np.exp(last_logits - last_logits.max(axis=1, keepdims=True))
    probs = e / e.sum(axis=1, keepdims=True)
    # sample from the distribution
    ix = rng.choice(vocab_size, p=probs[0])
    if ix == EOS_ID:
        break  # the model signaled the end of its own generation
    # shift the context window and track the samples
    context = context[1:] + [ix]
    out.append(ix)


print(tok.decode(out))

ct.
Jeffersu personarentian but The finedomitinghile thanefins in doubidarly, In�ent is promot is fined in fined independence of the fber  avoly avolution, thech; if youch; ’s ratherk home indental representwer seced Janeceminian but it is in the sale than Jane Austen’s.’s wreffers on the day. The efialization of her written find it is no leffersonoosed with The eflth or there abolutach maybte more subt not mean, a, Jane Austen’s for a fling that Jane maybtrs for the about’s for womr. Eresting is no in Mr.
k home in he know butw of Lady Catherscusetill inscth or there abolition the mother country,e Elizedrestigdfuss my int himts my, Jane play
